# Chat Mode — Integration Test

> **"How frequent were positive events (e.g. limit increases) in the last 18 months?"**

Tests the full chat pipeline: team construction → specialists → compare → synthesize → answer.

In [ ]:
%load_ext dotenv
%dotenv

%load_ext autoreload
%autoreload 2

import os, sys, json
import nest_asyncio
nest_asyncio.apply()

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if os.path.basename(os.getcwd()) != 'notebooks':
    PROJECT_ROOT = os.getcwd()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data', 'simulated')
PROFILE_DIR = os.path.join(PROJECT_ROOT, 'config', 'data_profiles')
PILLAR_DIR = os.path.join(PROJECT_ROOT, 'config', 'pillars')

# ═══════════════ CONFIG ═══════════════
SAFECHAIN_MODEL = "gpt-4.1"
CASE_ID = "CASE-00001"
PILLAR = "credit_risk"
MODE = "chat"
QUESTION = "How frequent were positive events (e.g. limit increases) in the last 18 months?"
# ═════════════════════════════════════

print(f'Case: {CASE_ID} | Pillar: {PILLAR} | Mode: {MODE}')
print(f'Question: {QUESTION}')

## 1. Load Data

In [ ]:
from data.gateway import SimulatedDataGateway
from data.catalog import DataCatalog
from tools.data_tools import init_tools

gw = SimulatedDataGateway.from_case_folders(DATA_DIR)
catalog = DataCatalog(profile_dir=PROFILE_DIR)
init_tools(gw, catalog)
gw.set_case(CASE_ID)

print(f'Tables: {gw.list_tables()}')

## 2. Preview Relevant Data

In [ ]:
# positive_events from model scores
rows = gw.query('model_scores')
if rows:
    for i, row in enumerate(rows[:3]):
        print(f'Row {i}: positive_events={row.get("positive_events", "N/A")}')
print()

# Customer tenure
tenure = gw.query('cust_tenure')
if tenure:
    print(f'Tenure: {tenure[0]}')

## 3. Connect LLM

In [ ]:
from gateway.safechain_adapter import SafeChainAdapter

try:
    from safechain.lcel import model as safechain_model
    llm = safechain_model(SAFECHAIN_MODEL)
    adapter = SafeChainAdapter(llm=llm, model_name=SAFECHAIN_MODEL)
    print(f'SafeChain adapter: model={SAFECHAIN_MODEL}')
except ImportError:
    from gateway.openai_adapter import OpenAIAdapter
    adapter = OpenAIAdapter(model='gpt-4.1')
    print(f'OpenAI adapter (SafeChain not available)')

## 4. Run Full Chat Pipeline

In [ ]:
from gateway.firewall_stack import FirewallStack
from log.event_logger import EventLogger
from agents.session_registry import SessionRegistry
from agents.general_specialist import GeneralSpecialist
from orchestrator.orchestrator import Orchestrator
from orchestrator.team import TeamConstructor
from orchestrator.chat_agent import ChatAgent
from config.pillar_loader import PillarLoader
from skills.domain.loader import load_domain_skill, list_domain_skills

logger = EventLogger(session_id=f'chat-{CASE_ID}', log_dir=os.path.join(PROJECT_ROOT, 'logs'))
pillar_loader = PillarLoader(pillar_dir=PILLAR_DIR)
pillar_config = pillar_loader.load(PILLAR) or {}
registry = SessionRegistry()

logger.log('session_start', {'case_id': CASE_ID, 'pillar': PILLAR, 'mode': MODE, 'question': QUESTION})
print('Pipeline initialized')

In [ ]:
# Team construction
logger.set_trace('q-001')
team_fw = FirewallStack(adapter=adapter, logger=logger)
team_constructor = TeamConstructor(firewall=team_fw, logger=logger)

selected = team_constructor.select_specialists(
    question=QUESTION, pillar=PILLAR,
    available_specialists=list_domain_skills(),
    active_specialists=registry.list_active(),
)
print(f'Selected: {selected}')

In [ ]:
# Run specialists
specialist_outputs = {}
for domain in selected:
    skill = load_domain_skill(domain)
    if skill is None:
        continue
    spec_config = pillar_loader.get_specialist_config(PILLAR, domain) or {}
    fw = FirewallStack(adapter=adapter, logger=logger)
    agent = registry.get_or_create(domain, PILLAR, skill, spec_config, fw, logger)
    output = agent.run(QUESTION, mode=MODE)
    specialist_outputs[domain] = output
    print(f'\n--- {domain} ---')
    print(f'Findings: {output.findings[:300]}')

print(f'\nSpecialists completed: {list(specialist_outputs.keys())}')

In [ ]:
# Compare
compare_fw = FirewallStack(adapter=adapter, logger=logger)
general = GeneralSpecialist(firewall=compare_fw, logger=logger)
review_report = general.compare(specialist_outputs, QUESTION)

print(f'Resolved: {len(review_report.resolved)}')
print(f'Open conflicts: {len(review_report.open_conflicts)}')
print(f'Insights: {review_report.cross_domain_insights}')

In [ ]:
# Synthesize + format
synth_fw = FirewallStack(adapter=adapter, logger=logger)
orchestrator = Orchestrator(synth_fw, logger, registry, PILLAR, pillar_config=pillar_config)
final = orchestrator.synthesize(specialist_outputs, review_report, QUESTION, MODE)

chat_agent = ChatAgent(firewall=synth_fw, logger=logger)
formatted = chat_agent.format_for_reviewer(final)

print('='*60)
print(f'Q: {QUESTION}')
print('='*60)
print(formatted)
print('='*60)

logger.log('final_output', {'question': QUESTION, 'specialists': final.specialists_consulted})
logger.clear_trace()

## 5. Follow-Up (Reuses Warm Specialists)

In [ ]:
FOLLOW_UP = "Are the positive events consistent with the bureau scores?"
logger.set_trace('q-002')

followup_outputs = {}
for domain in selected:
    skill = load_domain_skill(domain)
    spec_config = pillar_loader.get_specialist_config(PILLAR, domain) or {}
    fw = FirewallStack(adapter=adapter, logger=logger)
    agent = registry.get_or_create(domain, PILLAR, skill, spec_config, fw, logger)
    output = agent.run(FOLLOW_UP, mode=MODE)
    followup_outputs[domain] = output

review2 = general.compare(followup_outputs, FOLLOW_UP)
final2 = orchestrator.synthesize(followup_outputs, review2, FOLLOW_UP, MODE)

print('='*60)
print(f'FOLLOW-UP: {FOLLOW_UP}')
print('='*60)
print(chat_agent.format_for_reviewer(final2))
logger.clear_trace()

## 6. Session Info

In [ ]:
for a in registry.list_active():
    print(f'{a["domain"]}: {a["questions_answered"]} questions answered')

logger.log('session_end', {})
print('\nSession complete.')